# 04-2 학습 ② CLIP 파인튜닝 개선 (Colab, **GPU 필요: 런타임 → T4**)
04 1차에서 **파인튜닝이 모든 숫자를 떨어뜨렸다** (처음 보는 음식 주의놓침 8.5% → 11.5%, 검증 손실은 1에폭부터 계속 오름 = 과적합).
원인 3개를 하나씩 고친다.

| 1차 문제 | 04-2에서 바꾼 것 |
|---|---|
| 한 배치에 같은 음식 레시피가 여러 개인데 전부 "틀린 짝"으로 밀어냄 → 같은 음식끼리 멀어짐 | **같은 음식이면 정답 짝**으로 보는 손실 |
| 사진·글 모델 전체를 학습 → 원래 알던 음식 지식을 잊음 | **마지막 2층 + 출력층만** 학습, 나머지 고정 |
| 검증 손실로 저장 → 앱 성능과 안 맞음 | 검증 사진으로 **단계일치 − 주의놓침** 점수를 재서 에폭 선택 |
| (추가) | 원래 CLIP과 학습한 CLIP 가중치를 **섞어서**(WiSE-FT) 가장 좋은 비율 선택 |

**공정하게:** 에폭·섞는 비율은 **검증 사진**으로만 고르고, 시험(아는 음식·처음 보는 음식)은 마지막에 한 번만 잰다.
섞는 비율 0 = 원래 CLIP이라, 학습이 도움이 안 되면 그것도 숫자로 드러난다.

0~3은 04와 같다. 30~40분. 결과는 `찍먹_실험기록/표/clip2_결과.csv`, 모델은 `찍먹_실험기록/best/clip2/`.

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
IMG = f'{DATA}/images'                       # 03에서 모은 사진
RECIPES = f'{DATA}/table_recipes.parquet'    # 03에서 만든 레시피 (제목·재료)
EXP = find(TEAM, '찍먹_실험기록')
OUT, SAVE = f'{EXP}/표', f'{EXP}/best/clip2'   # 1차(best/clip)와 따로 저장
os.makedirs(SAVE, exist_ok=True)

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
assert torch.cuda.is_available(), '런타임 → 런타임 유형 변경 → T4 GPU'
dev = 'cuda'
print('준비 완료', torch.cuda.get_device_name())

## 1. 사진 ↔ 레시피 짝 만들기
- 사진 한 장 = 레시피 한 개의 대표 사진. 짝이 되는 글 = **레시피 제목 + 재료 목록**.
- 03에서 같은 사진이 반복된 가짜(`_fake_md5.csv`)는 뺀다.

In [ ]:
import numpy as np, pandas as pd
from knowledge.rules import allergen_flags, ingredients_text
from knowledge.judge import level            # 비율 → 주의 / 확인 필요 / 해당 없음 (앱과 같은 기준)

recipes = pd.read_parquet(RECIPES, columns=['recipe_idx', 'label', 'title', 'NER']).drop_duplicates(['label', 'recipe_idx'])
recipes['ing'] = recipes.NER.map(ingredients_text)
recipes['text'] = (recipes.title.str.strip() + '. ' + recipes.ing).str.slice(0, 300)   # CLIP 글 길이 한계(77토큰)

fake = set(pd.read_csv(f'{IMG}/_fake_md5.csv').md5)
photos = pd.concat([pd.read_parquet(p, columns=['recipe_idx', 'label', 'jpg', 'md5']) for p in sorted(glob.glob(f'{IMG}/*.parquet'))])
photos = (photos[~photos.md5.isin(fake)].merge(recipes[['recipe_idx', 'label', 'text']], on=['recipe_idx', 'label'])
          .sample(frac=1, random_state=0).reset_index(drop=True))
print(f'사진 {len(photos):,}장, 음식 {photos.label.nunique()}종 | 레시피 {len(recipes):,}개, 음식 {recipes.label.nunique()}종')
print(photos.text.iloc[0])

## 2. 음식 나누기 — "처음 보는 음식"을 진짜로 처음 보게
- **처음 보는 음식 20종**: ConvNeXt가 모르는 새 후보(74종) 중 사진 30장 이상인 것에서 무작위. 이 음식 사진은 **학습에 한 장도 안 씀**.
  이 음식들의 레시피 글은 검색 대상에 있다 (실제 앱도 레시피는 미리 다 넣어 두니까).
- **아는 음식**: 나머지. 레시피 단위로 학습 80 / 검증 10 / 시험 10 (같은 레시피가 학습·시험에 동시에 안 들어가게).
- 사진 30장 미만(김밥·떡볶이 등)은 숫자가 흔들려서 학습·평가에서 빼고, 검색 대상(방해꾼)으로만 둔다.

In [ ]:
known = set(open(f'{REPO}/modeling/classes.txt').read().split())
cnt = photos.label.value_counts()
FEW = sorted(cnt[cnt < 30].index)
new_ok = sorted(c for c in cnt[cnt >= 30].index if c not in known)
UNSEEN = sorted(np.random.default_rng(0).choice(new_ok, size=20, replace=False))
SEEN = sorted(c for c in cnt[cnt >= 30].index if c not in UNSEEN)

h = (photos.recipe_idx.values * 2654435761) % 100          # 레시피 번호로 정해지는 고정 분할
photos['split'] = np.select([photos.label.isin(UNSEEN), ~photos.label.isin(SEEN), h < 80, h < 90],
                            ['unseen', 'none', 'train', 'val'], 'test')
train, val = photos[photos.split == 'train'], photos[photos.split == 'val']
ev = {'아는 음식': photos[photos.split == 'test'].groupby('label').head(30),
      '처음 보는 음식': photos[photos.split == 'unseen'].groupby('label').head(100)}
sel = {'검증(선택용)': val.groupby('label').head(10)}   # 에폭·섞는 비율 고르기 전용. 시험 사진은 마지막에 한 번만 본다

# 검색 대상: 175종 레시피 (음식당 최대 300개). 시험 사진의 원래 레시피는 빼서 "자기 글 찾기" 반칙 방지
held = set(pd.concat([*ev.values(), *sel.values()]).recipe_idx)
index = (recipes[~recipes.recipe_idx.isin(held)].sample(frac=1, random_state=0)
         .groupby('label').head(300).reset_index(drop=True))
flags = allergen_flags(index.ing).reset_index(drop=True)       # 레시피 × 19항목 (룰)
ALLERGENS = list(flags.columns)
truth = pd.read_csv(f'{OUT}/food_table.csv', index_col='class')[ALLERGENS]   # 정답 = 그 음식 전체 레시피 통계

print('처음 보는 음식:', ', '.join(UNSEEN))
print(f'아는 음식 {len(SEEN)}종 | 학습 {len(train):,} · 검증 {len(val):,}장 | 사진 부족으로 제외 {len(FEW)}종: {", ".join(FEW)}')
print({k: len(v) for k, v in {**ev, **sel}.items()}, f'| 검색 대상 레시피 {len(index):,}개')

## 3. CLIP 준비
`openai/clip-vit-base-patch32` (서버 CPU에서도 빠른 크기). 사진과 글을 같은 공간의 512차원 벡터로 바꾸고, 가까울수록 비슷하다.

In [ ]:
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T
from transformers import CLIPModel, CLIPTokenizer

NAME = 'openai/clip-vit-base-patch32'
tok = CLIPTokenizer.from_pretrained(NAME)
MEAN, STD = [0.4815, 0.4578, 0.4082], [0.2686, 0.2613, 0.2758]
BIC = T.InterpolationMode.BICUBIC
tf_eval = T.Compose([T.Resize(224, interpolation=BIC), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1), interpolation=BIC), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
load = lambda b: Image.open(io.BytesIO(b)).convert('RGB')

def img_feat(m, px):
    return F.normalize(m.visual_projection(m.vision_model(pixel_values=px).pooler_output), dim=-1)

def txt_feat(m, ids, mask):
    return F.normalize(m.text_projection(m.text_model(input_ids=ids, attention_mask=mask).pooler_output), dim=-1)

def tokens(texts):
    return tok(list(texts), padding=True, truncation=True, max_length=77, return_tensors='pt')

@torch.no_grad()
def emb_text(m, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        t = tokens(texts[i:i + bs]).to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(txt_feat(m, t.input_ids, t.attention_mask).float())
    return torch.cat(out)

@torch.no_grad()
def emb_img(m, jpgs, bs=256):
    out = []
    for i in range(0, len(jpgs), bs):
        px = torch.stack([tf_eval(load(b)) for b in jpgs[i:i + bs]]).to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(img_feat(m, px).float())
    return torch.cat(out)

## 4. 평가 함수 + 파인튜닝 전 성능
사진 한 장마다 두 가지로 맞혀 본다.
- **이름 맞히기**: "a photo of pad thai, a type of food" 같은 음식 이름 문장 175개 중 가장 가까운 것 (CLIP 기본 사용법)
- **레시피 검색** (우리 방식): 가장 가까운 레시피 `K`개 → 많이 나온 음식이 답. 그리고 그 `K`개 레시피로 **알레르기 비율**을 계산해
  정답 음식의 표 비율과 비교 → `알레르기_오차`(평균 절대 오차), `단계일치`(주의/확인/해당없음이 같은 비율), `주의놓침`(정답이 주의인데 해당 없음으로 나온 비율, 낮을수록 좋음)

In [ ]:
K = 30
NAMES = sorted(truth.index)
lv = np.vectorize(level)

def evaluate(m, tag, sets=None):
    m.eval()
    Tx = emb_text(m, index.text.tolist())
    P = emb_text(m, [f'a photo of {c.replace("_", " ")}, a type of food.' for c in NAMES])
    lab = index.label.values
    Fl = torch.tensor(flags.values, dtype=torch.float32, device=dev)
    rows = []
    for name, d in (sets or ev).items():
        I, y = emb_img(m, d.jpg.tolist()), d.label.values
        zs = [[NAMES[j] for j in r] for r in (I @ P.T).topk(5).indices.tolist()]
        pred, est = [], []
        for c in range(0, len(I), 512):
            sim, idx = (I[c:c + 512] @ Tx.T).topk(K)
            est.append(Fl[idx].mean(1).cpu().numpy())
            for s_, i_ in zip(sim.tolist(), idx.tolist()):
                score = {}
                for s, j in zip(s_, i_):
                    score[lab[j]] = score.get(lab[j], 0) + s
                pred.append(sorted(score, key=score.get, reverse=True)[:5])
        est, tru = np.concatenate(est), truth.loc[y].values
        le, lt = lv(est), lv(tru)
        rows.append({'모델': tag, '시험': name, '장수': len(d),
                     '이름맞히기_top1': np.mean([p[0] == t for p, t in zip(zs, y)]),
                     '이름맞히기_top5': np.mean([t in p for p, t in zip(zs, y)]),
                     '레시피검색_top1': np.mean([p[0] == t for p, t in zip(pred, y)]),
                     '레시피검색_top5': np.mean([t in p for p, t in zip(pred, y)]),
                     '알레르기_오차': np.abs(est - tru).mean(),
                     '단계일치': (le == lt).mean(),
                     '주의놓침': (le[lt == '주의'] == '해당 없음').mean()})
    return pd.DataFrame(rows).round(3)

def score(r):
    """검증 점수: 단계일치는 높게, 주의놓침은 낮게 (앱에서 제일 중요한 두 숫자)."""
    r = r.iloc[0]
    return r['단계일치'] - r['주의놓침']

base = CLIPModel.from_pretrained(NAME).to(dev).eval()      # 마지막 섞기(WiSE-FT)에 쓰려고 지우지 않는다
res_before = evaluate(base, 'CLIP 그대로')
sel_base = evaluate(base, 'CLIP 그대로', sel)
print('검증 점수(그대로):', round(score(sel_base), 4))
res_before

## 5. 파인튜닝 2차
- **손실**: 배치 안에서 같은 음식 레시피는 모두 정답 (정답이 여러 개면 똑같이 나눠 가짐). 사진→글, 글→사진 양쪽.
- **고정**: 사진·글 모델의 마지막 2층과 출력층만 학습. 온도(`logit_scale`)도 고정.
- 3에폭, 에폭마다 검증 점수를 재서 가장 좋은 에폭 저장.

In [ ]:
from torch.utils.data import Dataset, DataLoader

LID = {c: i for i, c in enumerate(sorted(photos.label.unique()))}

class Pairs(Dataset):
    def __init__(self, d, tf): self.j, self.t, self.y, self.tf = d.jpg.tolist(), d.text.tolist(), d.label.map(LID).tolist(), tf
    def __len__(self): return len(self.j)
    def __getitem__(self, i): return self.tf(load(self.j[i])), self.t[i], self.y[i]

def collate(b):
    px, txt, y = zip(*b)
    t = tokens(txt)
    return torch.stack(px), t.input_ids, t.attention_mask, torch.tensor(y)

def clip_loss(m, px, ids, mask, y):
    i, t = img_feat(m, px), txt_feat(m, ids, mask)
    logits = m.logit_scale.exp() * i @ t.T
    pos = (y[:, None] == y[None, :]).float()               # 같은 음식 = 정답
    tgt = pos / pos.sum(1, keepdim=True)
    return (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.T, tgt)) / 2

def unfreeze_last(m, n=2):
    for p in m.parameters():
        p.requires_grad = False
    parts = [*m.vision_model.encoder.layers[-n:], m.vision_model.post_layernorm, m.visual_projection,
             *m.text_model.encoder.layers[-n:], m.text_model.final_layer_norm, m.text_projection]
    for part in parts:
        for p in part.parameters():
            p.requires_grad = True
    return [p for p in m.parameters() if p.requires_grad]

EPOCHS, BS, LR = 3, 128, 1e-5
torch.manual_seed(0)
model = CLIPModel.from_pretrained(NAME).to(dev)
params = unfreeze_last(model)
print(f'학습하는 파라미터 {sum(p.numel() for p in params) / 1e6:.1f}M / 전체 {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')
dl = DataLoader(Pairs(train, tf_train), BS, shuffle=True, drop_last=True, num_workers=2, collate_fn=collate)
vl = DataLoader(Pairs(val, tf_eval), BS, drop_last=True, num_workers=2, collate_fn=collate)
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.1)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')

best, log, t0 = score(sel_base), [], time.time()     # 원래 CLIP보다 나아야 저장
for ep in range(EPOCHS):
    model.train(); tl = 0
    for px, ids, mask, y in dl:
        with torch.autocast('cuda', dtype=torch.float16):
            loss = clip_loss(model, px.to(dev), ids.to(dev), mask.to(dev), y.to(dev))
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item()
    model.eval(); vls = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for px, ids, mask, y in vl:
            vls += clip_loss(model, px.to(dev), ids.to(dev), mask.to(dev), y.to(dev)).item()
    s = score(evaluate(model, f'{ep+1}에폭', sel))
    log.append({'epoch': ep + 1, 'train_loss': tl / len(dl), 'val_loss': vls / len(vl), '검증점수': s})
    saved = s > best
    if saved:
        best = s
        torch.save({k: v.half() for k, v in model.state_dict().items()}, f'{SAVE}/clip_ft2.pt')
    print(f'[{ep+1}/{EPOCHS}] train {tl/len(dl):.3f} | val {vls/len(vl):.3f} | 검증점수 {s:.4f}{" ← 저장" if saved else ""} | {(time.time()-t0)/60:.0f}분')
pd.DataFrame(log).to_csv(f'{SAVE}/train_log.csv', index=False)
TRAINED = os.path.exists(f'{SAVE}/clip_ft2.pt')
print('원래 CLIP보다 나은 에폭 있음' if TRAINED else '→ 어떤 에폭도 원래 CLIP보다 못함 (섞기도 건너뜀)')

## 6. 원래 CLIP과 섞기 (WiSE-FT) · 비율 고르기
가중치 = (1 − α) × 원래 CLIP + α × 학습한 CLIP. α = 0은 원래 그대로, 1은 학습한 그대로.
학습한 것의 장점은 살리고 잊어버린 지식은 되찾는 방법 (Wortsman et al., CVPR 2022). **α도 검증 사진으로만** 고른다.

In [ ]:
ALPHAS = [0, 0.25, 0.5, 0.75, 1.0]
rows, best_a = [], 0.0
if TRAINED:
    ft_sd = {k: v.float() for k, v in torch.load(f'{SAVE}/clip_ft2.pt').items()}
    base_sd = base.state_dict()
    mix = CLIPModel.from_pretrained(NAME).to(dev)
    for a in ALPHAS:
        mix.load_state_dict({k: ((1 - a) * base_sd[k].float() + a * ft_sd[k].to(dev)) if base_sd[k].is_floating_point() else base_sd[k]
                             for k in base_sd})
        r = evaluate(mix, f'α={a}', sel)
        rows.append(r.assign(검증점수=score(r)))
    sel_res = pd.concat(rows, ignore_index=True)           # α 하나당 한 줄
    best_a = ALPHAS[int(sel_res.검증점수.idxmax())]
    sel_res.to_csv(f'{OUT}/clip2_검증선택.csv', index=False, encoding='utf-8-sig')
    display(sel_res)
print('고른 α =', best_a)

## 7. 시험 (한 번만) · 1차와 비교
고른 에폭·α로 만든 모델을 아는 음식·처음 보는 음식 시험 사진에 처음으로 돌린다.

In [ ]:
if best_a > 0:
    mix.load_state_dict({k: ((1 - best_a) * base_sd[k].float() + best_a * ft_sd[k].to(dev)) if base_sd[k].is_floating_point() else base_sd[k]
                         for k in base_sd})
    final_model, tag = mix, f'CLIP 파인튜닝 2차 (α={best_a})'
    torch.save({k: v.half() for k, v in mix.state_dict().items()}, f'{SAVE}/clip_ft2_mix.pt')
    res_after = evaluate(final_model, tag)
else:
    final_model, tag = base, 'CLIP 그대로'
    res_after = res_before.iloc[0:0]       # 학습이 도움 안 됨 → 원래 CLIP이 결론

first = pd.read_csv(f'{OUT}/clip_결과.csv').query("모델 == 'CLIP 파인튜닝'").assign(모델='CLIP 파인튜닝 1차 (참고)')
res = pd.concat([res_before, first, res_after]).sort_values(['시험', '모델'])
res.to_csv(f'{OUT}/clip2_결과.csv', index=False, encoding='utf-8-sig')
print('앱에 쓸 모델:', tag)
res

1차(참고) 줄은 04에서 잰 숫자라 검색 대상이 아주 조금 다르다 (04-2는 검증 사진의 레시피도 검색 대상에서 뺐다). 비교의 기준은 같은 노트북에서 잰 `CLIP 그대로`.

**앱에 쓸 검색 목록 저장**: 고른 모델로 175종 레시피(음식당 최대 300개)를 벡터로 바꿔 둔다.

In [ ]:
app_index = recipes.sample(frac=1, random_state=0).groupby('label').head(300).reset_index(drop=True)
emb = emb_text(final_model, app_index.text.tolist()).half().cpu().numpy()
np.save(f'{SAVE}/index_emb.npy', emb)
app_index[['recipe_idx', 'label']].to_parquet(f'{SAVE}/index_meta.parquet', index=False)
open(f'{SAVE}/어떤_모델.txt', 'w').write(tag + ('' if best_a > 0 else ' (가중치 파일 없음: openai/clip-vit-base-patch32 그대로)'))
print(f'검색 목록 {emb.shape} 저장 → {SAVE}  ({tag})')

## 끝 — 결과 읽는 법
- `처음 보는 음식`에서 **2차가 `CLIP 그대로`보다 `단계일치`↑ `주의놓침`↓** → 학습 ② 효과 있음 (1차 실패 → 원인 분석 → 개선, 발표 흐름 그대로)
- 고른 α가 **0**이거나 2차가 그대로보다 못하면 → 결론은 "기본 CLIP을 레시피 검색에 쓰는 구조가 맞고, 우리 데이터 규모에선 파인튜닝 이득이 없다". 이것도 정직한 결과
- `clip2_검증선택.csv`: α별 검증 점수 (섞는 효과 그래프로 쓸 수 있음)

다 돌리면 Claude에게 **"04-2 다 돌렸어"**.